# Clase 3 — Prepara tus propios datos: imágenes de gatos y perros

**Manual asociado:** [manual/03_prepara_tus_datos.md](../manual/03_prepara_tus_datos.md)

## Resumen para la clase

- Los datos reales no llegan uniformes: tamaños distintos, formatos distintos.
- `cv2.imread` + `cv2.resize` estandarizan tus imágenes.
- Comprueba el balance de clases y mezcla (shuffle) tus datos antes de entrenar.
- Keras espera arrays con forma `(cantidad, alto, ancho, canales)`.
- `pickle` guarda datos ya procesados para no repetir el trabajo.

In [ ]:
# Descarga una version reducida y publica del dataset "cats_and_dogs"
# (la misma que usa Google en sus tutoriales oficiales de TensorFlow)
import tensorflow as tf
import pathlib

URL_DATASET = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
ruta_zip = tf.keras.utils.get_file("cats_and_dogs_filtered.zip", origin=URL_DATASET, extract=True)
directorio_base = pathlib.Path(ruta_zip).parent / "cats_and_dogs_filtered" / "train"

categorias = ["cats", "dogs"]
print("Directorio base:", directorio_base)
print("Contenido:", list(directorio_base.iterdir()))

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

ruta_ejemplo = directorio_base / "cats"
nombre_archivo = os.listdir(ruta_ejemplo)[0]
array_imagen = cv2.imread(str(ruta_ejemplo / nombre_archivo), cv2.IMREAD_GRAYSCALE)

plt.imshow(array_imagen, cmap="gray")
plt.title(f"Forma original: {array_imagen.shape}")
plt.show()

In [ ]:
IMG_SIZE = 50

array_redimensionado = cv2.resize(array_imagen, (IMG_SIZE, IMG_SIZE))
plt.imshow(array_redimensionado, cmap="gray")
plt.title(f"Redimensionada: {array_redimensionado.shape}")
plt.show()

In [ ]:
datos_entrenamiento = []


def crear_datos_entrenamiento():
    for categoria in categorias:
        ruta = directorio_base / categoria
        clase_num = categorias.index(categoria)  # cats=0, dogs=1

        for nombre_archivo in os.listdir(ruta):
            try:
                array_imagen = cv2.imread(str(ruta / nombre_archivo), cv2.IMREAD_GRAYSCALE)
                array_redimensionado = cv2.resize(array_imagen, (IMG_SIZE, IMG_SIZE))
                datos_entrenamiento.append([array_redimensionado, clase_num])
            except Exception:
                pass  # ignora archivos corruptos


crear_datos_entrenamiento()
print("Total de imagenes procesadas:", len(datos_entrenamiento))

In [ ]:
import random

random.shuffle(datos_entrenamiento)

conteo_por_clase = {0: 0, 1: 0}
for _, etiqueta in datos_entrenamiento:
    conteo_por_clase[etiqueta] += 1
print("Balance de clases (0=gato, 1=perro):", conteo_por_clase)

In [ ]:
import numpy as np
import pickle

X = []
y = []

for features, label in datos_entrenamiento:
    X.append(features)
    y.append(label)

X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y)

print("Forma final de X:", X.shape)

with open("X.pickle", "wb") as archivo_salida:
    pickle.dump(X, archivo_salida)
with open("y.pickle", "wb") as archivo_salida:
    pickle.dump(y, archivo_salida)

## Cierre de la clase

- `X.pickle` y `y.pickle` son la entrada de la clase 4, donde construyes tu primera CNN.
- **Comprueba antes de irte** que `X.shape` sea `(numero_de_imagenes, 50, 50, 1)`.